In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import logging
import time

# Logs directory config
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format= "%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

engine = create_engine("sqlite:///C:/Users/nikhi/data/inventory.db")
def ingest_db(df, table_name, engine):
    df.to_sql(table_name, con = engine, if_exists = 'replace', index = False)

def load_rawdata():
    start = time.time()
    chunk_size = 50000  
    
    for file in os.listdir('.'):
        if '.csv' in file:
            table_name = file[:-4]
            logging.info(f'Starting ingestion for {file} in db')
            print(f"Processing: {file}")
            
            first_chunk = True
            total_rows = 0
            
            # File ko chunks mein read karne ke liye loop
            for df_chunk in pd.read_csv(file, chunksize=chunk_size):
                total_rows += len(df_chunk)
                
                if first_chunk:
                    # Pehli chunk purani table ko 'replace' (delete) karegi
                    df_chunk.to_sql(table_name, con=engine, if_exists='replace', index=False)
                    first_chunk = False
                else:
                    # Baaki saari chunks usi table ke niche 'append' hoti jayengi
                    df_chunk.to_sql(table_name, con=engine, if_exists='append', index=False)
                    
            logging.info(f'Ingested {file} with total {total_rows} rows into table {table_name}')
            print(f"Finished {file}. Total rows: {total_rows}")
            
    end = time.time()
    total_time = (end - start) / 60
    logging.info('----------------Ingestion complete-------------------')
    logging.info(f'\nTotal time taken:{total_time} minutes')

if __name__ == '__main__':
    load_rawdata()


Processing: begin_inventory.csv
Finished begin_inventory.csv. Total rows: 206529
Processing: end_inventory.csv
Finished end_inventory.csv. Total rows: 224489
Processing: purchases.csv
Finished purchases.csv. Total rows: 2372474
Processing: purchase_prices.csv
Finished purchase_prices.csv. Total rows: 12261
Processing: sales.csv
Finished sales.csv. Total rows: 12825363
Processing: vendor_invoice.csv
